# Step 11: 서브에이전트 + 코디네이터 — 다중 에이전트 오케스트레이션

## 학습 목표

- **AgentTool**로 서브에이전트를 생성하는 방법을 이해합니다
- **부모 컨텍스트 포크(fork)** 패턴과 프롬프트 캐시 공유를 배웁니다
- **코디네이터 패턴**: Research -> Synthesis -> Implementation -> Verification 워크플로우를 구현합니다

> **서브에이전트란?**
> 메인 에이전트(부모)가 복잡한 작업을 만나면, 새로운 에이전트(자식)를 생성하여 작업을 위임합니다.
> 서브에이전트는 부모의 시스템 프롬프트를 상속받지만, 독립된 메시지 히스토리와 제한된 도구를 가집니다.
> 핵심 원칙: **코디네이터는 절대 직접 도구를 실행하지 않고, 오직 워커에게 지시만 합니다.**

---

## Claude Code 소스 분석

### 11-1. runAgent() — 서브에이전트 실행 코어 (src/tools/AgentTool/runAgent.ts:248+)

서브에이전트 실행의 핵심 함수입니다. 부모와 동일한 에이전트 루프를 재활용하되, 독립된 컨텍스트에서 실행됩니다:

```typescript
// src/tools/AgentTool/runAgent.ts:248+ (핵심 로직 간소화)

async function runAgent(params: {
  prompt: string,
  parentSystem: string,           // 부모의 시스템 프롬프트
  agentDefinition: AgentDefinition,
  tools?: string[],               // 사용 가능한 도구 제한
}): Promise<string> {
  // 1. 시스템 프롬프트 병합 (부모 + 에이전트 고유)
  const system = mergeSystemPrompts(
    params.parentSystem,
    params.agentDefinition.systemPrompt
  )

  // 2. 도구 필터링 — 서브에이전트는 제한된 도구만 사용
  const filteredTools = params.tools
    ? allTools.filter(t => params.tools.includes(t.name))
    : allTools

  // 3. ★ 에이전트 루프 실행 (부모와 동일한 queryLoop 재활용)
  const result = await queryLoop({
    messages: [{ role: 'user', content: params.prompt }],
    system,
    tools: filteredTools,
    maxTurns: parentMaxTurns / 2,  // 서브에이전트는 턴 제한 절반
  })

  return extractFinalText(result)
}
```

### 11-2. forkSubagent() — 컨텍스트 포크 (src/tools/AgentTool/forkSubagent.ts:18-71)

서브에이전트를 생성할 때 가장 중요한 것은 **프롬프트 캐시 공유**입니다:

```typescript
// src/tools/AgentTool/forkSubagent.ts:18-71 (핵심 간소화)

function forkSubagent(parentContext: Context): SubagentContext {
  // ★ 부모의 시스템 프롬프트를 그대로 상속
  // → API의 prompt caching 덕분에, 부모와 동일한 시스템 프롬프트는
  //   캐시에서 가져오므로 추가 비용이 거의 없음
  const forkedSystem = parentContext.systemPrompt

  // ★ 메시지 히스토리는 비우기 (독립 실행)
  // → 서브에이전트는 부모의 대화 내용을 모르고, 오직 프롬프트만 받음
  const messages: Message[] = []

  // ★ 도구는 필터링하여 제한
  const tools = filterTools(parentContext.tools, agentDef.allowedTools)

  return { system: forkedSystem, messages, tools }
}
```

**설계 포인트:**
- 시스템 프롬프트 상속 → 프롬프트 캐시 효율 극대화 (비용 절감)
- 메시지 히스토리 비움 → 서브에이전트는 "깨끗한 상태"에서 시작
- 도구 제한 → 서브에이전트가 불필요한 작업을 하지 못하게 방지

### 11-3. 내장 에이전트 (src/tools/AgentTool/builtInAgents.ts)

Claude Code는 용도별 서브에이전트를 사전 정의해둡니다:

```typescript
// src/tools/AgentTool/builtInAgents.ts (간소화)

const builtInAgents = {
  general: {
    description: "범용 서브에이전트",
    systemPrompt: "복잡한 작업을 분할하여 처리합니다",
    // tools: 모든 도구 사용 가능
  },
  explore: {
    description: "코드베이스 탐색 전문",
    systemPrompt: "파일을 검색하고, 구조를 분석합니다. 수정하지 마세요.",
    tools: ["Bash", "Read", "Grep", "Glob"],  // ★ 읽기 전용 도구만
  },
  plan: {
    description: "작업 계획 수립",
    systemPrompt: "목표를 분석하고 단계별 계획을 만듭니다. 직접 수정하지 마세요.",
    tools: ["Bash", "Read", "Grep", "Glob"],  // ★ 읽기 전용 도구만
  },
}
```

### 11-4. 코디네이터 모드 (src/coordinator/coordinatorMode.ts)

코디네이터는 복잡한 작업을 4단계 워크플로우로 분해합니다:

```
Research (탐색)        → explorer 에이전트가 코드베이스를 조사
    ↓
Synthesis (분석)       → planner 에이전트가 조사 결과를 종합하여 계획 수립
    ↓
Implementation (구현)  → general 에이전트가 계획에 따라 코드 수정
    ↓
Verification (검증)    → explorer 에이전트가 변경 결과를 확인
```

**핵심 규칙: 코디네이터는 절대 직접 도구를 실행하지 않습니다.** 오직 서브에이전트에게 작업을 지시하고, 결과를 종합합니다. 이것은 "관리자는 직접 코딩하지 않고, 팀원에게 일을 분배한다"는 현실 세계의 패턴을 그대로 반영한 것입니다.

---

## Python으로 구현하기

### 구현 1: AgentDefinition과 내장 에이전트

`mini_claude/tools/agent_tool.py`에 구현된 서브에이전트 시스템을 살펴봅니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.tools.agent_tool import (
    AgentDefinition,
    AgentTool,
    BUILT_IN_AGENTS,
    spawn_agent,
)

# --- 1. 내장 에이전트 정의 확인 ---
print("=== 내장 에이전트 목록 ===")
for name, defn in BUILT_IN_AGENTS.items():
    tools = ", ".join(defn.tools) if defn.tools else "(모든 도구)"
    print(f"  {name:20s} — {defn.description}")
    print(f"    도구: {tools}")
    print(f"    시스템: {defn.system_prompt[:60]}...")
    print()

# --- 2. AgentDefinition 구조 ---
print("=== AgentDefinition 필드 ===")
explorer = BUILT_IN_AGENTS["explorer"]
print(f"  name:          {explorer.name}")
print(f"  description:   {explorer.description}")
print(f"  tools:         {explorer.tools}")
print(f"  model:         {explorer.model}")  # None이면 부모와 동일한 모델 사용

### 구현 2: spawn_agent()와 컨텍스트 포크

`spawn_agent()`는 Claude Code의 `forkSubagent()` + `runAgent()`에 대응합니다.
부모의 시스템 프롬프트를 상속하면서 독립된 에이전트 루프를 실행합니다.

In [ ]:
# --- 1. spawn_agent() — 시뮬레이션 모드 (runner=None) ---
# runner가 없으면 실제 LLM 호출 없이 시뮬레이션 결과를 반환합니다.

result = await spawn_agent(
    definition=BUILT_IN_AGENTS["explorer"],
    prompt="mini_claude 프로젝트의 디렉토리 구조를 분석해주세요",
    parent_system="당신은 유능한 AI 코딩 어시스턴트입니다.",
)
print("=== spawn_agent (시뮬레이션) ===")
print(result)

# --- 2. 커스텀 에이전트 정의 ---
reviewer = AgentDefinition(
    name="code_reviewer",
    description="코드 품질을 검토하는 전문 리뷰어",
    system_prompt=(
        "당신은 코드 리뷰 전문가입니다. "
        "버그, 보안 취약점, 성능 문제, 가독성을 검토합니다. "
        "각 이슈에 심각도(높음/중간/낮음)를 부여하세요."
    ),
    tools=["Read", "Grep", "Glob"],  # 읽기 전용 도구만
)

result = await spawn_agent(
    definition=reviewer,
    prompt="agent.py 파일의 코드 품질을 리뷰해주세요",
    parent_system="당신은 유능한 AI 코딩 어시스턴트입니다.",
)
print("\n=== 커스텀 에이전트 (시뮬레이션) ===")
print(result)

# --- 3. AgentTool — LLM이 호출하는 도구 인터페이스 ---
agent_tool = AgentTool(parent_system="AI 코딩 어시스턴트")

print(f"\n=== AgentTool ===")
print(f"  name: {agent_tool.name}")
print(f"  description: {agent_tool.description[:100]}...")

# AgentTool을 통한 서브에이전트 생성
result = await agent_tool.call({
    "agent": "planner",
    "prompt": "REST API 서버를 구축하는 계획을 세워주세요",
})
print(f"\n=== AgentTool 실행 결과 ===")
print(result.data)

---

## 실습: 코디네이터 패턴 — 다단계 워크플로우 시뮬레이션

코디네이터 패턴을 시뮬레이션합니다. 코디네이터는 직접 도구를 실행하지 않고,
각 단계에 맞는 서브에이전트를 생성하여 작업을 위임합니다.

In [ ]:
import asyncio

# --- 코디네이터 패턴 시뮬레이션 ---
# Claude Code의 coordinatorMode.ts를 Python으로 재현합니다.
# 실제로는 각 단계에서 LLM을 호출하지만, 여기서는 시뮬레이션합니다.

async def coordinator_workflow(task: str) -> dict[str, str]:
    """
    코디네이터 워크플로우 — 4단계로 복잡한 작업을 처리
    
    핵심 규칙: 코디네이터는 절대 직접 도구를 실행하지 않습니다.
    오직 서브에이전트에게 지시하고, 결과를 종합합니다.
    """
    results = {}
    
    # ── Phase 1: Research (탐색) ──
    print("=" * 60)
    print("[Phase 1] Research — explorer 에이전트가 코드베이스를 조사합니다")
    print("=" * 60)
    
    research_result = await spawn_agent(
        definition=BUILT_IN_AGENTS["explorer"],
        prompt=f"다음 작업을 위해 코드베이스를 조사해주세요: {task}\n"
               "관련 파일, 구조, 의존성을 파악하세요.",
        parent_system="AI 코딩 어시스턴트",
    )
    results["research"] = research_result
    print(f"  결과: {research_result[:100]}...")
    
    # ── Phase 2: Synthesis (분석/계획) ──
    print(f"\n{'=' * 60}")
    print("[Phase 2] Synthesis — planner 에이전트가 계획을 수립합니다")
    print("=" * 60)
    
    synthesis_result = await spawn_agent(
        definition=BUILT_IN_AGENTS["planner"],
        prompt=f"조사 결과를 바탕으로 실행 계획을 수립해주세요.\n"
               f"작업: {task}\n"
               f"조사 결과: {research_result[:200]}",
        parent_system="AI 코딩 어시스턴트",
    )
    results["synthesis"] = synthesis_result
    print(f"  결과: {synthesis_result[:100]}...")
    
    # ── Phase 3: Implementation (구현) ──
    print(f"\n{'=' * 60}")
    print("[Phase 3] Implementation — general 에이전트가 계획을 실행합니다")
    print("=" * 60)
    
    impl_result = await spawn_agent(
        definition=BUILT_IN_AGENTS["general_purpose"],
        prompt=f"다음 계획에 따라 작업을 수행해주세요.\n"
               f"계획: {synthesis_result[:200]}",
        parent_system="AI 코딩 어시스턴트",
    )
    results["implementation"] = impl_result
    print(f"  결과: {impl_result[:100]}...")
    
    # ── Phase 4: Verification (검증) ──
    print(f"\n{'=' * 60}")
    print("[Phase 4] Verification — explorer 에이전트가 결과를 검증합니다")
    print("=" * 60)
    
    verify_result = await spawn_agent(
        definition=BUILT_IN_AGENTS["explorer"],
        prompt=f"작업 결과를 검증해주세요.\n"
               f"원래 작업: {task}\n"
               f"구현 결과: {impl_result[:200]}",
        parent_system="AI 코딩 어시스턴트",
    )
    results["verification"] = verify_result
    print(f"  결과: {verify_result[:100]}...")
    
    return results

# 코디네이터 워크플로우 실행
results = await coordinator_workflow(
    "mini_claude 프로젝트에 로깅 시스템을 추가해주세요"
)

print(f"\n{'=' * 60}")
print("[코디네이터] 완료 — 4개 단계 모두 실행됨")
print(f"  단계별 결과 키: {list(results.keys())}")
print("=" * 60)

---

## 핵심 설계 원칙 정리

### 1. 컨텍스트 포크 = 시스템 프롬프트 상속 + 메시지 히스토리 초기화
서브에이전트는 부모의 "성격"(시스템 프롬프트)은 물려받지만, "기억"(대화 히스토리)은 비우고 시작합니다. API의 프롬프트 캐싱 덕분에 시스템 프롬프트 상속은 거의 무료입니다.

### 2. 도구 제한으로 안전한 위임
탐색 에이전트는 읽기 전용 도구만, 구현 에이전트는 쓰기 도구도 허용. 역할에 따른 도구 제한이 안전하고 예측 가능한 서브에이전트 실행을 보장합니다.

### 3. 코디네이터는 지시만 하고 직접 실행하지 않는다
코디네이터 패턴의 핵심 규칙입니다. 코디네이터가 직접 도구를 실행하면 책임이 불분명해지고, 에러 복구가 어려워집니다.

### 4. 에이전트 루프 재활용
서브에이전트를 위한 별도의 실행 엔진이 아니라, 부모와 동일한 `agent_loop()`를 재활용합니다. 설정만 다르게 주면 됩니다. 이것이 코드 복잡성을 크게 줄입니다.

---

## 다음 Step 미리보기

서브에이전트가 독립적으로 작업하는 패턴을 배웠습니다. 이제 에이전트를 외부 시스템과 연결해봅시다.

**Step 12**에서는 Claude Code의 **Bridge 시스템**을 분석하여:
- **양방향 WebSocket 통신**으로 IDE/웹 UI와 에이전트를 연결하는 방법
- **메시지 프로토콜**: user, assistant, control_request, control_response
- **Echo dedup 패턴**과 원격 권한 처리

이 패턴들을 Python으로 구현합니다.